# Experiment record: Task B, Run 1

**Status:** `documented_recipe_result`.

**Result / finding:** Rebuild of the winning TAPT recipe; documentation records OOF 0.6013 and competition validation 0.5922 for that recipe. No log uniquely bound to this notebook copy was recovered.

[Individual result record and log-derived metric summaries](../../../results/experiments/task_b/01b_rebuild_winner.json)

**Source:** `notebooks/task_b/01b_rebuild_winner.ipynb`, export commit `9282c844fd29`.
Original notebook SHA-256: `2aef3e0de712f24660973dda1d36c01625e7c8cb5ae805cc6fbdbf238000b8b6`.

## Historical source snapshot

The original code cells below are preserved unchanged for methodological inspection; saved outputs and execution counters are cleared. Original repository/bootstrap paths and experimental APIs are retained as historical source. They have **not** been migrated or GPU-validated against the curated release. Use the two top-level Kaggle launchers for the supported final-system pipeline.

Historical prose below can contain superseded plans, “ready” states, or older “final” selections. The outcome above and the official final scores in `results/final_scores.json` take precedence. Source-at-export does not necessarily equal source-at-execution.



# HASTIKA Task B -- rebuild the winning submission

Only for the case where last night's output never published. This reruns the one arm
that won, `b_tapt` at five folds, and nothing else. About 2 hours on a T4 against the
8.5 hours the full sweep took.

Seeds and split seeds are fixed, so this reproduces the same model that scored
**0.6013** rather than rolling the dice again.

Sidebar: Accelerator `GPU T4 x2` or `GPU P100`, Internet on. Save Version -> Save & Run All.


In [ ]:
import os, subprocess, sys, pathlib
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git","-C",WORK,"pull","--ff-only"], check=True)
else:
    subprocess.run(["git","clone","-q","-b","task-b","--depth","1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK); os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True); print("cwd:", os.getcwd())

subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    """Streams, tees, and raises. Nothing here is allowed to fail quietly."""
    print(f"$ {' '.join(cmd)}", flush=True)
    fh = open(log,"w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Domain-adapt MuRIL (~25 min)

This is the step that was worth +2.9 points over stock MuRIL. Watch the held-out
perplexity fall across the 8 epochs.


In [ ]:
run([sys.executable,"-u","-m", "hastika.task_b.tapt","--out","artifacts/runs/tapt-muril"],
    log="artifacts/logs/tapt.log")


## 2. Five folds on the adapted encoder (~90 min)


In [ ]:
run([sys.executable,"-u","-m", "hastika.task_b.train","--tag","b_tapt_5f",
     "--model","artifacts/runs/tapt-muril","--folds","5","--seeds","42","--epochs","6"],
    log="artifacts/logs/b_tapt_5f.log")


## 3. Package

`make_submission.py` refuses to write unless the header is `id,label`, every id in
`multiclass_validation_inputs.csv` appears exactly once, and every label is one of the
six. Upload the zip it names to the Task B phase.


In [ ]:
ZIP = "/kaggle/working/b_tapt_5f.zip"
run([sys.executable,"-m", "hastika.common.submission","--task","b",
     "--pred","artifacts/runs/b_tapt_5f/predictions.csv","--out",ZIP])
run(["cp","artifacts/logs/b_tapt_5f.log","/kaggle/working/"])
assert os.path.exists(ZIP)
run(["unzip","-l",ZIP])

# the OOF score and the per-class report, so you can confirm it reproduced
import re
t = pathlib.Path("artifacts/logs/b_tapt_5f.log").read_text()
for line in t.splitlines():
    if "OOF macro-F1" in line or "macro avg" in line:
        print(line)
print("\nDownload b_tapt_5f.zip from the Output tab.")
